<a href="https://colab.research.google.com/github/Eugenia311/agente-vinted/blob/main/agente_vinted.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Asistente de anuncios para Vinted con IA

Herramienta que escribe anuncios de Vinted a partir de los datos básicos de una prenda, usando la API de Gemini. Es la segunda fase de mi proyecto [Análisis de mis ventas en Vinted](https://github.com/Eugenia311/analisis-vinted), donde descubrí que lo que no se vendía tenía muy pocas visitas.

**Cómo funciona:** unas instrucciones de sistema definen las normas del asistente:
- No inventar: solo usa los datos que recibe.
- Tono cercano, en primera persona y sin adjetivos de entusiasmo.
- Ideas para combinar la prenda.
- Títulos con las palabras que usaría un comprador al buscar.

**Ejemplo:** la calidad del anuncio depende de los datos que recibe. Con «Bufanda, marca Primark, estado muy bueno», el título fue «Bufanda Primark». Con color, estampado y medidas, fue «Bufanda de cuadros azul crema marrón Primark».

**Detalles técnicos:**
- Python con la librería `google-genai`, en Google Colab.
- La clave de API se guarda en los secretos de Colab, nunca en el código.
- Si un modelo está saturado o sin cuota, prueba automáticamente con el siguiente.

**Herramientas:** Python, API de Gemini, Google Colab, con Claude como apoyo para el aprendizaje.

In [1]:
!pip install -q -U google-genai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.63.1 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.6.1 which is incompatible.


In [2]:
from google import genai
from google.colab import userdata

cliente = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

for modelo in cliente.models.list():
    if "flash" in modelo.name:
        print(modelo.name)

models/gemini-2.5-flash
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash


In [3]:
from google.genai import types

instrucciones = """
Eres una asistente que ayuda a una vendedora particular a escribir anuncios para Vinted.

Recibirás los datos de una prenda o artículo: tipo, marca, talla, estado y cualquier detalle extra (color, material, defectos, por qué lo vende).

Escribe en texto plano, sin asteriscos ni ningún otro formato.

Escribe el anuncio siguiendo estas normas:

1. Descripción real: cuenta solo lo que te han dado. No inventes materiales, medidas ni detalles. Si hay defectos, menciónalos con naturalidad; la sinceridad genera confianza.Tampoco inventes el motivo de la venta ni detalles personales: si no te los dan, no los menciones.
2. Tono cercano: escribe como una persona que vende algo suyo, no como una tienda. Frases sencillas, en español de España, sin exagerar. No uses adjetivos de opinión o entusiasmo (precioso, genial, increíble, perfecto, ideal…). En su lugar, describe hechos concretos: el color, el estilo, el estado o para qué sirve.
3. Que el comprador vea posibilidades: sugiere en una o dos frases cómo combinarlo o en qué ocasiones usarlo. Escribe en primera persona y, si te dan algún detalle personal (por qué lo vende, cuánto lo ha usado), inclúyelo. Puedes cerrar con una frase amable invitando a preguntar.

Formato de respuesta:
- Título: corto y claro. Incluye las palabras que un comprador escribiría al buscarlo: tipo de prenda, marca, color y talla si los tienes. Nada de adjetivos vacíos como «precioso» o «increíble».
- Descripción: entre 3 y 5 frases.
"""

prenda = "Sandalias verdes, marca Topshop, estado muy bueno"

modelos = ["gemini-3.5-flash", "gemini-3.8-flash", "gemini-3.1-flash-lite"]

for modelo in modelos:
    try:
        respuesta = cliente.models.generate_content(
            model=modelo,
            contents=prenda,
            config=types.GenerateContentConfig(system_instruction=instrucciones),
        )
        print(f"Modelo usado: {modelo}\n")
        print(respuesta.text)
        break
    except Exception:
        print(f"{modelo} no disponible, pruebo el siguiente...")
        import time


Modelo usado: gemini-3.5-flash

Título: Sandalias verdes Topshop

Descripción:
Vendo estas sandalias de color verde de la marca Topshop. Se encuentran en muy buen estado. Combinan bien con vestidos sencillos o con pantalones vaqueros para añadir un toque de color a la ropa. Sirven tanto para llevar a diario como para salir a tomar algo. Si tienes alguna duda, me puedes escribir sin problema.


In [4]:
import time

modelos = ["gemini-3.5-flash", "gemini-3.8-flash", "gemini-3.1-flash-lite"]

def escribir_anuncio(prenda):
    for modelo in modelos:
        try:
            respuesta = cliente.models.generate_content(
                model=modelo,
                contents=prenda,
                config=types.GenerateContentConfig(system_instruction=instrucciones),
            )
            return respuesta.text.replace("**", "")
        except Exception as error:
            print(f"{modelo} no disponible: {error}")
    return "Ningún modelo disponible ahora mismo. Inténtalo más tarde."


sin_vender = [
    "Sandalias verdes metalizadas, marca Topshop, estado muy bueno, talla EU 36 / UK 3 / US 5.5, tacón transparente geométrico",
    "Falda plisada, tul plumeti, marca Stradivarius, estado muy bueno, color negro, cintura sin estirar: 33 cm (66 cm de contorno), largo total: 85 cm, cintura elástica",
    "Bufanda, marca Primark, estado muy bueno, estampado de cuadros en tonos azul, crema y marrón, largo 200 cm, ancho 36 cm",
    "Pantalón palazzo, sin marca, nuevo con etiquetas",
]

for prenda in sin_vender:
    print("=" * 60)
    print(prenda)
    print("=" * 60)
    print(escribir_anuncio(prenda))
    print()
    time.sleep(10)

Sandalias verdes metalizadas, marca Topshop, estado muy bueno, talla EU 36 / UK 3 / US 5.5, tacón transparente geométrico
Título: Sandalias verdes metalizadas Topshop talla 36

Descripción:
Vendo estas sandalias verdes con acabado metalizado de la marca Topshop en talla 36, en muy buen estado. Tienen un tacón transparente con diseño geométrico. Quedan bien para salir de noche o acudir a algún evento combinadas con prendas en tonos neutros o un vestido negro. Escríbeme si tienes cualquier duda.

Falda plisada, tul plumeti, marca Stradivarius, estado muy bueno, color negro, cintura sin estirar: 33 cm (66 cm de contorno), largo total: 85 cm, cintura elástica
Título: Falda plisada de tul plumeti Stradivarius negra

Descripción: 
Vendo esta falda plisada de tul plumeti de la marca Stradivarius en color negro. Está en muy buen estado y lleva la cintura elástica, que mide 33 cm en plano (66 cm de contorno sin estirar) y tiene un largo total de 85 cm. Se puede combinar con unas zapatillas y un